<a href="https://colab.research.google.com/github/SalesforceAIResearch/SlackAgents/blob/main/docs/docs/examples/llm/openai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using OpenAILLM

This notebook demonstrates how to use the `OpenAILLM` offered by `SlackAgents` to generate text in single and multi-turn conversations and function calls.

If you're opening this Notebook on colab, you will probably need to install `SlackAgents`. You can do this by running the following cell:

```python
!pip install slackagents
```

In [ ]:
!pip install slackagents

## Setup
First, let's import the necessary modules and set up our API key.


In [1]:
import os
from slackagents.llms.openai import OpenAILLM, BaseLLMConfig
from pprint import pprint
# Set up the OpenAI API key (you can also set this as an environment variable)
os.environ["OPENAI_API_KEY"] = "sk-..."

## Creating a Configuration

Now, we'll create a `BaseLLMConfig` object to configure our language model parameters. The configuration class inherits from `BaseLLMConfig` and has the following attributes:

- model: Optional[str] - Controls the OpenAI model used (default: None)
- temperature: float - Controls the randomness of the output (default: 0)
- api_key: Optional[str] - OpenAI API key to be used (default: None)
- max_tokens: int - Controls how many tokens are generated (default: 3000)
- top_p: float - Controls the diversity of words (default: 0)
- top_k: int - Controls the diversity of words (default: 1)

Note: There are additional parameters specific to Openrouter and Ollama, which are not typically used for standard OpenAI configurations.

In [2]:
config = BaseLLMConfig(
  model="gpt-4o",
  temperature=0.7,
  max_tokens=150,
  top_p=1.0
)

## Initialize the OpenAILLM instance


In [3]:
llm = OpenAILLM(config)

## Simple Conversation
Let's start with a simple conversation to test our setup.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's the capital of France?"}
]
response = llm.chat_completion(messages)
pprint(response)

## Function Calling
Now, let's try a more complex interaction using tools. We'll define a weather tool and ask about the weather.

In [5]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_current_weather",
      "description": "Get the current weather in a given location",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "The city and state, e.g. San Francisco, CA"
            },
          "unit": {
            "type": "string",
            "enum": ["celsius", "fahrenheit"]
            }
        },
      "required": ["location"]
      }
    }
  }
]

In [ ]:
# Previous conversations
messages_with_tool = [
  {"role": "system", "content": "You are a helpful assistant."},
  {"role": "user", "content": "What's the weather like in New York?"}
]
# Add a tool to the conversation
response_with_tool = llm.chat_completion(messages_with_tool, tools=tools)
pprint(response_with_tool)

## Structured Response
Structured Outputs is a feature that ensures the model will always generate responses that adhere to your supplied JSON Schema, so you don't need to worry about the model omitting a required key, or hallucinating an invalid enum value. 

Note that this feature is only available for the most recent OpenAI's GPT-4o model, like `gpt-4o-2024-08-06`.

In [7]:
config = BaseLLMConfig(
  model="gpt-4o-2024-08-06",
  temperature=0.7,
  max_tokens=512,
  top_p=1.0
)
llm = OpenAILLM(config)

### Defining the JSON Schema of the Response

In [8]:
from pydantic import BaseModel
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

### Returning the Structured Response

In [ ]:
messages = [
      {"role": "system", "content": "Extract the event information."},
      {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
]
response_with_schema = llm.chat_completion(messages, response_format=CalendarEvent)
pprint(response_with_schema)

### Chain of Thought (CoT) Reasoning 
You can ask the model to output an answer in a structured, step-by-step way, to guide the user through the solution, which in LLM literature, is termed as CoT reasoning strategy.

In [10]:
class Step(BaseModel):
    explanation: str
    output: str

class CoTReasoning(BaseModel):
    steps: list[Step]
    final_answer: str

In [ ]:
messages = [
        {"role": "system", "content": "You are a helpful math tutor. Guide the user through the solution step by step."},
        {"role": "user", "content": "how can I solve 8x + 7 = -23"}
]

response_with_cot = llm.chat_completion(messages, response_format=CoTReasoning)
pprint(response_with_cot)

## Conclusion

These examples demonstrate how to use different response formats with the `OpenAILLM` class. The text format is suitable for general conversations, while the JSON format is excellent for structured data and can be easily parsed and used in further processing.